**Intelligent Vision Systems · Lab 7 · Reading & Implementing Research Papers**
Companion to **Lecture 7 — Reading & Implementing Research Papers**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/01-pytorch-for-papers.ipynb) &nbsp; [Lecture 7 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/07_Paper_Implementation.html#/pytorch-in-10-minutes) &nbsp; [All Lab 7 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-7-papers) &nbsp; [Setup guide: Colab & Antigravity](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/09_paper_implementation_setup.html)

# Lab 7A · PyTorch for Paper Readers

| ⏱ Time | 💻 Runs on | 🎯 You will build |
|---|---|---|
| 45–60 min | Google Colab (CPU is enough, GPU is faster) or Google Antigravity / VS Code on your laptop | the 6 PyTorch skills every paper implementation needs: tensors, shapes, conv arithmetic, reshaping, nn.Module, the training loop |

**How to work.** Run the cells from top to bottom with **Shift + Enter**. Cells marked **🖐 Your turn** have a `# TODO`:
write a few lines, run the cell, then run the **✅ check** below it. A 💡 means "not yet": read the hint, fix, run again.
Stuck for more than 10 minutes? Open the **Hint** boxes, look at the matching lecture slide, or ask the AI assistant to
*explain* (not write) the step. 

Papers describe models with **figures, shapes and equations**. PyTorch is how we turn them into running code.
This notebook teaches only what you need for Lab 7B–7H. No prior PyTorch needed, but you must know basic Python and NumPy
(Lecture 3 and the [NumPy tutorial](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/06_numpy.html)).

In [ ]:
# Setup: run this first (Colab already has PyTorch; on your laptop see requirements.txt).
# !pip install -q torch torchvision matplotlib scikit-learn opencv-python

import os, urllib.request
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_7"
for f in ["lab7_checks.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("lab files ready")

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab7_checks as C          # auto-checks + helpers for this lab

C.seed_everything(0)
device = C.get_device()
print("PyTorch", torch.__version__, "· device:", device)

## 1 · A tensor is a NumPy array that can live on a GPU and remember gradients

In every vision paper an image batch has the shape **(B, C, H, W)**: batch size, channels, height, width.
(OpenCV/NumPy images are (H, W, C); PyTorch puts the channels **first**.)

In [ ]:
x = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
print("x =", x)
print("shape:", x.shape, "· dtype:", x.dtype, "· device:", x.device)
print("x.T @ x =\n", x.T @ x)                     # matrix product, like NumPy
print("as NumPy:", x.numpy().sum(), "| back:", torch.from_numpy(np.ones(3)))

### 🖐 Your turn 1: make a batch of images
Create `img`: a batch of **8** RGB images of size **32 × 32**, filled with random numbers in [0, 1).
<details><summary><b>Hint 1</b></summary>

`torch.rand(...)` takes the shape as arguments.

</details>
<details><summary><b>Hint 2</b></summary>

Order: (batch, channels, height, width).

</details>

In [ ]:
img = None   # TODO
print(img.shape if img is not None else "make img first")

In [ ]:
C.check_batch(img)

In [ ]:
# Indexing works like NumPy. To SHOW an image with matplotlib we need (H, W, C) again: permute.
first = img[0]                    # (3, 32, 32)
plt.figure(figsize=(2, 2)); plt.imshow(first.permute(1, 2, 0)); plt.axis("off"); plt.title("img[0]"); plt.show()
print("one pixel, all channels:", img[0, :, 5, 7])

## 2 · Conv arithmetic: the formula behind every shape in a paper

For a convolution (or pooling) with input size $n$, kernel $k$, stride $s$, padding $p$, dilation $d$:

$$ n_{out} = \left\lfloor \frac{n + 2p - d\,(k-1) - 1}{s} \right\rfloor + 1 $$

This one line explains U-Net's 572 → 570, ResNet's 224 → 112 and ViT's 224 → 14 × 14 patches.

### 🖐 Your turn 2: write `conv_out`
<details><summary><b>Hint 1</b></summary>

Use integer floor division `//`.

</details>
<details><summary><b>Hint 2</b></summary>

With d = 1 the term d·(k − 1) is just k − 1. Then + 1 at the end.

</details>

### 📄 ↔ 💻 Paper ↔ Code: every "output size" in a paper comes from one formula

<table>
<tr><th width="50%" align="left">📄 Paper · He et al. 2016, Table 1</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_r_table1.png" width="100%" alt="pc_r_table1.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre># out = floor((n + 2p - k) / s) + 1        (dilation 1)
# Check ResNet&#x27;s Table 1, column &quot;output size&quot;, with YOUR conv_out():
#   conv1: 7x7, 64, stride 2, padding 3      224 -&gt; ?   (Table 1: 112 x 112)
#   3x3 max pool, stride 2, padding 1        112 -&gt; ?   (Table 1: 56 x 56)
#   conv3_1: 3x3, stride 2, padding 1         56 -&gt; ?   (Table 1: 28 x 28)</pre>
</td></tr>
</table>

In [ ]:
def conv_out(n, k, s=1, p=0, d=1):
    # TODO: return the output size (an int)
    return n

In [ ]:
C.check_conv_out(conv_out)

### 🖐 Your turn 3: a layer that halves the image and makes 16 channels
Make an `nn.Conv2d` that maps **(8, 3, 32, 32) → (8, 16, 16, 16)** with a **3 × 3** kernel. Use `conv_out` to choose the stride and padding.
<details><summary><b>Hint 1</b></summary>

`nn.Conv2d(in_channels, out_channels, kernel_size, stride=..., padding=...)`

</details>
<details><summary><b>Hint 2</b></summary>

Halving needs stride 2. Try padding 0 and 1 in `conv_out(32, 3, 2, p)`: which gives 16?

</details>

In [ ]:
layer = None   # TODO: nn.Conv2d(...)

In [ ]:
C.check_conv_layer(layer)
print("output shape:", layer(img).shape, "· weights:", tuple(layer.weight.shape), "· bias:", tuple(layer.bias.shape))

## 3 · Reshape vs permute: images → tokens

Transformers want a **sequence of tokens** (B, N, D). Turning a feature map (B, C, H, W) into tokens is the first line of
ViT, DETR and many other papers: every pixel becomes a token whose features are its C channel values.

⚠️ `reshape` only re-reads the numbers in memory order; `permute`/`transpose` really moves axes. Mixing them up is the
**#1 silent bug** in paper implementations: the shape is right, the numbers are wrong, the model trains badly and you never know why.

In [ ]:
t = torch.arange(2 * 3).reshape(2, 3)
print(t)
print("reshape(3, 2):\n", t.reshape(3, 2), "\n<- rows re-cut, NOT a transpose")
print("transpose:\n", t.T)

### 🖐 Your turn 4: `to_tokens`
Write `to_tokens(x)` that turns (B, C, H, W) into (B, H·W, C), so that `tokens[b, i]` holds the C channel values of pixel *i*.
<details><summary><b>Hint 1</b></summary>

`x.flatten(2)` gives (B, C, H·W).

</details>
<details><summary><b>Hint 2</b></summary>

Then swap the last two axes with `.transpose(1, 2)`.

</details>

In [ ]:
def to_tokens(x):
    # TODO
    return x

In [ ]:
C.check_to_tokens(to_tokens)

## 4 · `nn.Module`: how every model in every paper is written

A model is a class with two methods: `__init__` creates the layers (they hold the learnable weights), `forward` says how
data flows through them. PyTorch computes gradients for you.

### 🖐 Your turn 5: `TinyMLP`
Build **Linear(2 → 32) → ReLU → Linear(32 → 32) → ReLU → Linear(32 → 2)**. Count the parameters by hand first
(a Linear(a → b) has a·b weights + b biases), then let the check count them.
<details><summary><b>Hint 1</b></summary>

Put the layers in `nn.Sequential(...)` inside `__init__` and call it in `forward`.

</details>

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: self.net = nn.Sequential(...)

    def forward(self, x):
        # TODO
        return x

model = TinyMLP()

In [ ]:
C.check_tiny_mlp(model)
print(model)

## 5 · The training loop: five lines you will write in every lab

```
for each batch:
    optimizer.zero_grad()          # 1. forget the old gradients
    pred = model(x)                # 2. forward
    loss = loss_fn(pred, y)        # 3. how wrong are we?
    loss.backward()                # 4. gradients of the loss w.r.t. every weight
    optimizer.step()               # 5. move every weight a little downhill
```
We test it on the "two moons" toy dataset (two interleaved half-circles).

In [ ]:
from sklearn.datasets import make_moons
Xn, yn = make_moons(600, noise=0.12, random_state=0)
X, y = torch.tensor(Xn, dtype=torch.float32), torch.tensor(yn)
plt.figure(figsize=(4, 3)); plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=8); plt.title("two moons"); plt.show()

### 🖐 Your turn 6: fill in the loop
<details><summary><b>Hint 1</b></summary>

Use `loss_fn = nn.CrossEntropyLoss()`: it takes raw scores (B, 2) and labels (B,).

</details>
<details><summary><b>Hint 2</b></summary>

The order matters: zero_grad → forward → loss → backward → step.

</details>

In [ ]:
model = TinyMLP()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()
history = []
for epoch in range(300):
    # TODO: the five lines (use the whole dataset as one batch)
    loss = torch.tensor(0.0)
    history.append(loss.item())
plt.figure(figsize=(4, 2.5)); plt.plot(history); plt.xlabel("epoch"); plt.ylabel("loss"); plt.show()

In [ ]:
C.check_trained(model, X, y)

In [ ]:
# What did it learn? Colour every point of the plane by the predicted class.
gx, gy = torch.meshgrid(torch.linspace(-1.5, 2.5, 200), torch.linspace(-1, 1.5, 200), indexing="xy")
with torch.no_grad():
    zz = model(torch.stack([gx.ravel(), gy.ravel()], 1)).argmax(1).reshape(gx.shape)
plt.figure(figsize=(4, 3)); plt.contourf(gx, gy, zz, alpha=0.3, cmap="coolwarm")
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=8); plt.title("decision boundary"); plt.show()

## 6 · `trace_shapes`: your best friend when implementing a paper

`C.trace_shapes(model, x)` prints the output shape of every layer. When you implement a paper, run it after **every**
block you add and compare with the sizes in the paper's figure or table. Here is a tiny CNN of the kind you saw in Lecture 3:

In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 8 * 8, 10))
C.trace_shapes(cnn, img);

## 7 · The paper → PyTorch dictionary

| In the paper | In PyTorch |
|---|---|
| "a 3 × 3 convolution with 64 filters" | `nn.Conv2d(c_in, 64, 3, padding=1)` |
| "unpadded convolution" | `padding=0` |
| "2 × 2 max pooling with stride 2" | `nn.MaxPool2d(2)` |
| "up-convolution", "transposed convolution" | `nn.ConvTranspose2d(c, c // 2, 2, stride=2)` |
| "fully connected / linear / projection, W x + b" | `nn.Linear(d_in, d_out)` |
| "batch normalisation", "layer normalisation" | `nn.BatchNorm2d(c)`, `nn.LayerNorm(d)` |
| "softmax over …" | `x.softmax(dim=…)` (which dim? read carefully!) |
| "concatenate" / "add" (residual) | `torch.cat([a, b], dim=1)` / `a + b` |
| $QK^\top$ | `q @ k.transpose(-2, -1)` |
| "dropout with rate 0.1" | `nn.Dropout(0.1)` |
| "Adam with β₁ = 0.9, β₂ = 0.98" | `torch.optim.Adam(params, lr, betas=(0.9, 0.98))` |

## ✍️ Your turn (answer in a markdown cell)
1. A 224 × 224 image goes through a 7 × 7 conv with stride 2 and padding 3, then a 3 × 3 max-pool with stride 2 and padding 1. What size comes out? (ResNet's stem: check with `conv_out`.)
2. Why is `x.reshape(B, H*W, C)` wrong for (B, C, H, W) input even though the shape is right?
3. In the training loop, what happens if you forget `optimizer.zero_grad()`? (You will meet this bug again in Lab 7G.)

**Next:** [Lab 7B · Attention from scratch](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/02-attention-from-scratch.ipynb)

### 📄 ↔ 💻 Paper ↔ Code: read real paper sentences as PyTorch

<table>
<tr><th width="50%" align="left">📄 Paper · Ronneberger et al. 2015, §2 · Vaswani et al. 2017, Eq. (2)</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_u_arch.png" width="100%" alt="pc_u_arch.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_ffn.png" width="100%" alt="pc_t_ffn.png"><br></td>
<td width="50%" valign="top">
<b>💻 Code</b>
<pre># U-Net §2, sentence by sentence:
&quot;two 3x3 convolutions (unpadded)&quot;        nn.Conv2d(cin, cout, 3)          # padding=0
&quot;each followed by a ReLU&quot;                nn.ReLU()
&quot;2x2 max pooling ... with stride 2&quot;      nn.MaxPool2d(2)
&quot;2x2 convolution (up-convolution)&quot;       nn.ConvTranspose2d(c, c // 2, 2, stride=2)
&quot;a concatenation with ...&quot;               torch.cat([a, b], dim=1)          # channels
&quot;a 1x1 convolution ... classes&quot;          nn.Conv2d(64, n_classes, 1)
# Transformer Eq. (2):
&quot;max(0, x W1 + b1) W2 + b2&quot;              lin2(torch.relu(lin1(x)))</pre>
</td></tr>
</table>